In [1]:
!pip install docling PyMuPDF

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 21.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 629.0/629.0 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.8/261.8 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 145.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 131.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 29.9 MB/s eta 0:00:00
   ━

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%%writefile parsing_colab.py
"""
PDF Parser per Google Colab (L4 o A100)
Istruzioni:
1. Carica la cartella `todo` sul tuo Google Drive.
2. Apri un Colab, vai su Runtime -> Cambia tipo di runtime -> L4 o A100 GPU.
3. Monta il tuo Google Drive nel notebook.
4. Installa le librerie: !pip install docling PyMuPDF
5. Esegui questo script!
"""

import json
import os
import re
import sys
import time
import shutil
import gc
from pathlib import Path
import fitz

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

# ─── Configurazione per Colab ──────────────────────────────────
# Metti qui il percorso della tua cartella "todo" su Google Drive
COLAB_INPUT_DIR = "/content/drive/MyDrive/todo"
COLAB_WORKING_DIR = "/content/processed"

RAW_DIR = Path(COLAB_INPUT_DIR)
PROCESSED_DIR = Path(COLAB_WORKING_DIR)

LEVEL_MAP = {
    "A": "bambini",
    "B": "medie",
    "C": "superiori",
    "D": "universitari",
}

# ─── Utility ──────────────────────────────────────────────────────
def parse_filename(filename: str) -> dict:
    stem = Path(filename).stem
    match = re.match(r"^([A-D](?:\s*,\s*[A-D])*)\s*-\s*(.+)$", stem)
    if match:
        levels_str = match.group(1)
        title = match.group(2).strip()
        levels = [l.strip() for l in levels_str.split(",")]
    else:
        levels = ["unknown"]
        title = stem

    return {
        "difficulty_levels": levels,
        "difficulty_labels": [LEVEL_MAP.get(l, "sconosciuto") for l in levels],
        "title": title,
    }

def parse_topic_folder(folder_name: str) -> dict:
    match = re.match(r"^TOPIC\s+(\d+)\s*-\s*(.+)$", folder_name)
    if match:
        return {
            "topic_id": int(match.group(1)),
            "topic_name": match.group(2).strip(),
        }
    return {"topic_id": 0, "topic_name": folder_name}

# ─── Main ─────────────────────────────────────────────────────────
def main():
    if not RAW_DIR.exists():
        print(f"ERRORE: Cartella {RAW_DIR} non trovata.")
        print("Assicurati di aver montato Google Drive e che il percorso sia corretto!")
        sys.exit(1)

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

    pdf_files = []
    for parent_dir in RAW_DIR.iterdir():
        if parent_dir.is_dir():
            pdf_files.extend(list(parent_dir.glob("**/*.pdf")))

    if not pdf_files:
         print("Nessun PDF trovato in", RAW_DIR)
         sys.exit(1)

    print(f"Totale PDF da processare su Colab: {len(pdf_files)}")

    print("---------------------------------------------------------")
    print("AVVIO INIZIALIZZAZIONE MODELLI DOCLING (Uso singola GPU)")
    print("---------------------------------------------------------")

    pipeline_options = PdfPipelineOptions()
    pipeline_options.accelerator_options.num_threads = 4  # Colab ha CPU migliori
    pipeline_options.ocr_batch_size = 4                   # A100/L4 possono gestire batch più grandi
    pipeline_options.do_formula_enrichment = True
    pipeline_options.do_ocr = True

    try:
        converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
            }
        )
        print("[+] Modelli caricati con successo sulla GPU!")
    except Exception as e:
        print(f"[-] ERRORE INIZIALIZZAZIONE MODELLI: {e}")
        return

    success = 0
    t_start_all = time.time()

    for idx, pdf_path in enumerate(pdf_files, 1):
        topic_folder = pdf_path.parent.name
        topic_info = parse_topic_folder(topic_folder)
        file_info = parse_filename(pdf_path.name)

        out_dir = PROCESSED_DIR / topic_folder
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{pdf_path.stem}.json"

        if out_path.exists():
            print(f"[{idx}/{len(pdf_files)}] SKIP: {pdf_path.name} già processato.")
            continue

        print(f"\n[{idx}/{len(pdf_files)}] Inizio: {pdf_path.name}")

        try:
            t_start = time.time()
            with fitz.open(str(pdf_path)) as doc:
                num_pages = len(doc)

            # Convertiamo a chunk più larghi (L4/A100 hanno molta più VRAM)
            CHUNK_SIZE = 15
            text_chunks = []

            for start_page in range(1, num_pages + 1, CHUNK_SIZE):
                end_page = min(start_page + CHUNK_SIZE - 1, num_pages)
                print(f"  -> Elaborando pagine {start_page}-{end_page}/{num_pages}...")

                conv_result = converter.convert(str(pdf_path), page_range=(start_page, end_page))
                text_chunks.append(conv_result.document.export_to_markdown())

                del conv_result
                gc.collect()

            text = "\n\n".join(text_chunks)
            elapsed = time.time() - t_start

            result = {
                "source_file": f"{topic_folder}/{pdf_path.name}",
                "topic_id": topic_info["topic_id"],
                "topic_name": topic_info["topic_name"],
                "difficulty_levels": file_info["difficulty_levels"],
                "difficulty_labels": file_info["difficulty_labels"],
                "title": file_info["title"],
                "num_pages": num_pages,
                "text_markdown": text,
            }

            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)

            print(f"  ✓ Finito in {elapsed:.1f}s: {pdf_path.name}")
            success += 1

        except Exception as e:
            print(f"  ✗ ERRORE su {pdf_path.name}: {e}")

    t_end_all = time.time() - t_start_all
    print(f"\n{'='*60}")
    print(f"TUTTO COMPLETATO in {t_end_all / 60:.1f} minuti ({success}/{len(pdf_files)} PDF)")

    print("\nCreazione archivio .zip dei risultati in corso...")
    shutil.make_archive("/content/processed_results", 'zip', PROCESSED_DIR)
    print("✓ Finito! Ora scarica 'processed_results.zip' dal pannello file di Colab.")

if __name__ == "__main__":
    main()


Writing parsing_colab.py


In [4]:
!python parsing_colab.py

Totale PDF da processare su Colab: 1
---------------------------------------------------------
AVVIO INIZIALIZZAZIONE MODELLI DOCLING (Uso singola GPU)
---------------------------------------------------------
[+] Modelli caricati con successo sulla GPU!

[1/1] Inizio: D - Stellar Structure and Evolution (Rudolf Kippenhahn, Alfred Weigert etc.).pdf
  -> Elaborando pagine 1-15/596...
[INFO] 2026-07-07 07:07:37,617 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-07 07:07:37,620 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-07-07 07:07:37,622 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.9.1/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-07 07:07:38,664 [RapidOCR] download_file.py:82: Download size: 13.83MB
100% 14.5M/14.5M [00:00<00:00, 71.7MiB/s]
[INFO] 2026-07-07 07:07:38,868 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-pack

In [8]:

!cp -r /content/processed /content/drive/MyDrive/salvataggio_elaborati




